<a href="https://colab.research.google.com/github/agmCorp/opinologo/blob/main/20260621_opinologo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Opinólogo
### Motor de Argumentacion Formal sobre LLMs

**Opinólogo** convierte cualquier pregunta controversial en un analisis argumentativo formal,
produciendo una conclusion *justificada* (o *refutada*) con trazabilidad completa.

---

### Como funciona?

| Paso | Componente | Que hace |
|------|-----------|----------|
| 1 | **C1 -- Generador** | LLM produce argumentos PRO en Lenguaje Natural Controlado (CNL) |
| 2 | **C2 -- Abogado del Diablo** | Mismo LLM, rol diferente: genera los mejores ataques posibles |
| 3 | **Loop adversarial** | C1 y C2 se alternan N rondas construyendo el grafo de argumentos |
| 4 | **C3 -- Parser CNL** | Convierte texto semi-estructurado a estructuras formales ASPIC+ |
| 5 | **C4 -- Motor Dung** | Calcula la *grounded extension*: que argumentos sobreviven todos los ataques |
| 6 | **C5 -- Narrador** | Traduce el resultado formal a una explicacion en lenguaje natural trazable |

---

### Arquitectura

```
            [Pregunta]
                |
                v
+---------------------------------+
|  C1: Generador                  |  LLM - argumentos PRO en CNL (IDs: A1, A2, ...)
+---------------+-----------------+
                | loop adversarial (max N rondas)
                v
+---------------------------------+
|  C2: Abogado del Diablo         |  LLM - ataques + contra-args en CNL (IDs: B1, B2, ...)
+---------------+-----------------+
                |
                v
+---------------------------------+
|  C3: Parser CNL -> ASPIC+       |  Deterministico - texto -> estructuras formales
+---------------+-----------------+
                |
                v
+---------------------------------+
|  C4: Motor ASPIC+/Dung          |  Deterministico - computa grounded extension
+---------------+-----------------+
                |
                v
+---------------------------------+
|  C5: Narrador                   |  LLM - explicacion trazable en lenguaje natural
+---------------+-----------------+
                |
                v
[Conclusion + arbol de argumentos + grafo JSON]
```

---

### Lenguaje Natural Controlado (CNL)

```
# Argumento defeasible (rebatible -- usar por defecto)
ARG A1: "premisa1", "premisa2" => "conclusion"

# Argumento estricto (logicamente necesario)
ARG A2: "premisa1" -> "conclusion"

# Con excepcion explicita
ARG A3: "premisa1" => "conclusion" SALVO "excepcion"

# Rebuttal: A contradice la conclusion de B
A1 REFUTA B1

# Undercut: A invalida la regla defeasible de B
A2 SOCAVA B2

# Conclusion que se defiende
CONCLUSION: "texto de la conclusion"
```


## Paso 1 -- Instalacion

Ejecuta esta celda **una sola vez** al inicio de la sesion Colab.
Luego ejecuta **Runtime > Ejecutar todo** (Ctrl+F9) para cargar todos los modulos.


In [ ]:
# Dependencias para todos los proveedores en la nube
!pip install -q openai google-genai anthropic huggingface_hub

# Dependencias adicionales para HuggingFaceLocalClient (Opcion F - GPU local)
# Solo necesaria si vas a usar modelos locales en Colab con GPU T4/A100.
!pip install -q transformers bitsandbytes accelerate

print("Dependencias instaladas.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.1 MB/s eta 0:00:00
Dependencias instaladas.


## Modulo 1 -- Estructuras de Datos

Define los tipos de datos centrales del sistema ASPIC+/Dung.

| Clase | Descripcion |
|-------|-------------|
| `Argument` | Un argumento formal: premisas + conclusion + tipo de regla |
| `Attack` | Relacion de ataque entre dos argumentos (rebuttal o undercut) |
| `ArgumentGraph` | El grafo completo acumulado durante el loop adversarial |
| `EvaluationResult` | Resultado del motor Dung: que argumentos sobrevivieron |
| `PipelineResult` | Resultado final completo: grafo + evaluacion + explicacion |


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Literal


@dataclass
class Argument:
    # Argumento formal en ASPIC+.
    # id:         identificador unico, ej: "A1", "B3"
    # premises:   lista de premisas en lenguaje natural
    # conclusion: lo que se infiere de las premisas
    # rule_type:  "defeasible" (=>) rebatible por defecto
    #             "strict"    (->) inferencia logicamente necesaria
    # exception:  excepcion explicita (clausula SALVO), si aplica
    id: str
    premises: list[str]
    conclusion: str
    rule_type: Literal["strict", "defeasible"]
    exception: str | None = None


@dataclass
class Attack:
    # Relacion de ataque entre dos argumentos.
    # attacker_id: ID del argumento que ataca
    # target_id:   ID del argumento atacado
    # attack_type: "rebuttal" (REFUTA) = conclusion contradice conclusion del objetivo
    #              "undercut" (SOCAVA) = invalida la regla defeasible del objetivo
    attacker_id: str
    target_id: str
    attack_type: Literal["rebuttal", "undercut"]


@dataclass
class ArgumentGraph:
    # Grafo de argumentacion completo.
    # Acumula todos los argumentos y ataques generados durante el loop adversarial.
    # to_cnl() lo serializa para pasarlo como contexto al LLM.
    arguments: dict[str, Argument] = field(default_factory=dict)
    attacks: list[Attack] = field(default_factory=list)
    target_conclusion: str = ""

    def to_cnl(self) -> str:
        lines = []
        for arg in self.arguments.values():
            premises_str = ", ".join(f'"{p}"' for p in arg.premises)
            arrow = "->" if arg.rule_type == "strict" else "=>"
            line = f'ARG {arg.id}: {premises_str} {arrow} "{arg.conclusion}"'
            if arg.exception:
                line += f' SALVO "{arg.exception}"'
            lines.append(line)
        for attack in self.attacks:
            verb = "REFUTA" if attack.attack_type == "rebuttal" else "SOCAVA"
            lines.append(f"{attack.attacker_id} {verb} {attack.target_id}")
        if self.target_conclusion:
            lines.append(f'CONCLUSION: "{self.target_conclusion}"')
        return "\n".join(lines)


@dataclass
class EvaluationResult:
    # Resultado de la evaluacion formal (semantica de Dung).
    conclusion: str
    justified: bool
    supporting_argument_id: str | None
    grounded_extension: list[str]
    defeated: list[str]
    undecidable: list[str]


@dataclass
class PipelineResult:
    # Resultado completo del pipeline.
    explanation: str
    graph: ArgumentGraph
    evaluation: EvaluationResult


print("Modulo 1 OK: Argument, Attack, ArgumentGraph, EvaluationResult, PipelineResult")


Modulo 1 OK: Argument, Attack, ArgumentGraph, EvaluationResult, PipelineResult


## Modulo 2 -- Parser CNL -> ASPIC+

Convierte texto CNL (generado por el LLM) a estructuras de datos formales.
Lineas que no coinciden con ningun patron se ignoran silenciosamente.


In [ ]:
import re

_ARG_RE = re.compile(
    r'ARG\s+(\w+)\s*:\s*((?:"[^"]*",?\s*)+)[=>]+\s*"([^"]*)"'
)
_EXCEPT_RE   = re.compile(r'SALVO\s+"([^"]*)"')
_ATTACK_RE   = re.compile(r'(\w+)\s+(REFUTA|SOCAVA)\s+(\w+)')
_CONCL_RE    = re.compile(r'CONCLU(?:SI[OO]N|SION)\s*:\s*"?([^"\n]+)"?')


def parse_cnl_block(block: str) -> tuple[list, list, str | None]:
    # Parsea un bloque CNL y devuelve (argumentos, ataques, conclusion_o_None).
    arguments: list[Argument] = []
    attacks: list[Attack] = []
    conclusion: str | None = None

    for line in block.splitlines():
        line = line.strip()

        m = _ARG_RE.match(line)
        if m:
            arg_id       = m.group(1)
            raw_premises = m.group(2)
            concl_text   = m.group(3)
            premises     = re.findall(r'"([^"]*)"', raw_premises)
            rule_type    = "strict" if ("->" in line and "=>" not in line) else "defeasible"
            exception    = None
            exc_m = _EXCEPT_RE.search(line)
            if exc_m:
                exception = exc_m.group(1)
            arguments.append(Argument(
                id=arg_id, premises=premises, conclusion=concl_text,
                rule_type=rule_type, exception=exception,
            ))
            continue

        m = _ATTACK_RE.match(line)
        if m:
            attacker, verb, target = m.group(1), m.group(2), m.group(3)
            attacks.append(Attack(
                attacker_id=attacker,
                target_id=target,
                attack_type="rebuttal" if verb == "REFUTA" else "undercut",
            ))
            continue

        m = _CONCL_RE.match(line)
        if m:
            conclusion = m.group(1).strip().strip('"')

    return arguments, attacks, conclusion


def validate_block(block: str, known_ids: set) -> list[str]:
    errors: list[str] = []
    for line in block.splitlines():
        line = line.strip()
        if line.count('"') % 2 != 0:
            errors.append(f"Comilla sin cerrar: {line[:60]}")
        m = _ATTACK_RE.match(line)
        if m and known_ids:
            attacker, target = m.group(1), m.group(3)
            if attacker not in known_ids:
                errors.append(f"ID atacante desconocido: {attacker}")
            if target not in known_ids:
                errors.append(f"ID objetivo desconocido: {target}")
    return errors


def extract_tagged_block(raw: str, tag: str) -> str:
    open_tag, close_tag = f"<{tag}>", f"</{tag}>"
    start = raw.find(open_tag)
    end   = raw.find(close_tag)
    if start == -1 or end == -1:
        return raw
    return raw[start + len(open_tag):end].strip()


print("Modulo 2 OK: parse_cnl_block, validate_block, extract_tagged_block")


Modulo 2 OK: parse_cnl_block, validate_block, extract_tagged_block


## Modulo 3 -- Motor de Argumentacion (Semantica de Dung)

Implementa el algoritmo de **grounded extension** de Dung (1995).

Un argumento se acepta si todos sus atacantes estan rechazados.
Un argumento se rechaza si al menos un atacante esta aceptado.
Si ninguna condicion aplica: indecidible (empate genuino).


In [ ]:
from __future__ import annotations


def grounded_extension(graph: ArgumentGraph) -> EvaluationResult:
    # Computa la grounded extension del grafo de argumentacion.
    # Punto fijo monotónico O(n^2).
    all_ids = set(graph.arguments.keys())

    attackers: dict[str, set] = {id_: set() for id_ in all_ids}
    for attack in graph.attacks:
        if attack.target_id in attackers and attack.attacker_id in all_ids:
            attackers[attack.target_id].add(attack.attacker_id)

    accepted: set[str] = set()
    rejected: set[str] = set()

    changed = True
    while changed:
        changed = False
        for arg_id in all_ids:
            if arg_id in accepted or arg_id in rejected:
                continue
            if all(a in rejected for a in attackers[arg_id]):
                accepted.add(arg_id)
                changed = True
        for arg_id in all_ids:
            if arg_id in accepted or arg_id in rejected:
                continue
            if any(a in accepted for a in attackers[arg_id]):
                rejected.add(arg_id)
                changed = True

    undecidable = sorted(all_ids - accepted - rejected)

    # Soporte: 1) coincidencia exacta con target_conclusion
    # 2) cualquier A-arg aceptado (los A-args son el lado PRO-conclusion)
    supporting_id: str | None = next(
        (id_ for id_ in sorted(accepted)
         if graph.arguments[id_].conclusion == graph.target_conclusion),
        None,
    )
    if supporting_id is None:
        supporting_id = next(
            (id_ for id_ in sorted(accepted) if id_.startswith("A")),
            None,
        )

    return EvaluationResult(
        conclusion=graph.target_conclusion,
        justified=supporting_id is not None,
        supporting_argument_id=supporting_id,
        grounded_extension=sorted(accepted),
        defeated=sorted(rejected),
        undecidable=undecidable,
    )


print("Modulo 3 OK: grounded_extension")


Modulo 3 OK: grounded_extension


## Modulo 4 -- Prompts de Sistema

| Prompt | Rol | Proposito |
|--------|-----|-----------|
| `GENERATOR_SYSTEM` | C1 | Construir argumentos PRO en CNL |
| `GENERATOR_INITIAL_TEMPLATE` | C1 | Primera llamada: generar desde cero |
| `GENERATOR_RESPONSE_TEMPLATE` | C1 | Rondas siguientes: rebatir ataques del Diablo |
| `DEVIL_SYSTEM` | C2 | Generar los mejores ataques posibles |
| `DEVIL_TEMPLATE` | C2 | Cada invocacion del Abogado del Diablo |
| `NARRATOR_SYSTEM` | C5 | Traducir el resultado formal a lenguaje natural |
| `NARRATOR_TEMPLATE` | C5 | Cada invocacion del Narrador |


In [ ]:
GENERATOR_SYSTEM = """You are an expert argumentation analyst. Construct the strongest arguments supporting a defensible conclusion.

Output format -- CNL (Controlled Natural Language):
  ARG [ID]: "[premise1]", "[premise2]" => "[conclusion]"   (defeasible, use by default)
  ARG [ID]: "[premise1]" -> "[conclusion]"                  (strict, only if logically necessary)
  ARG [ID]: "[premise1]" => "[conclusion]" SALVO "[exception]"
  CONCLUSION: "[the conclusion you defend]"

Rules:
- Use IDs A1, A2, A3, ... (never reuse an ID already in the graph)
- Premises and conclusions are affirmative statements in double quotes
- At most 5 premises per argument -- simpler arguments are stronger
- Generate 3-6 arguments
- End with CONCLUSION: "[conclusion]"
- Wrap output in <ARGUMENTOS>...</ARGUMENTOS> tags"""


GENERATOR_INITIAL_TEMPLATE = """Problem: {problem}

Generate arguments defending the most defensible conclusion."""


GENERATOR_RESPONSE_TEMPLATE = """Problem: {problem}

Current argument graph:
{current_graph}

Generate additional arguments to rebut the attacks (B-IDs) above. Continue the A-ID sequence from the graph.

CRITICAL: For each new argument that rebuts a B-argument, you MUST declare the attack:
  A_ID REFUTA B_ID   or   A_ID SOCAVA B_ID

Example:
  ARG A6: "studies show remote productivity matches office productivity" => "remote work does not reduce output"
  A6 REFUTA B3

Every new argument must be followed by the attack declaration it supports."""


DEVIL_SYSTEM = """You are a rigorous devil's advocate. Find the strongest possible attacks against the given arguments.

Output format -- same CNL rules as the generator, plus attack declarations:
  [ID_A] REFUTA [ID_B]    (rebuttal: A's conclusion contradicts B's)
  [ID_A] SOCAVA [ID_B]    (undercut: A attacks B's defeasible rule)

CRITICAL: Every argument you declare MUST be followed immediately by at least one attack declaration.
An argument with no REFUTA or SOCAVA line is invalid -- it has no effect on the graph.

Example of correct format:
  ARG B1: "remote work reduces face-to-face collaboration" => "remote work harms innovation"
  B1 REFUTA A2
  ARG B2: "enforcing office attendance increases commute stress" => "mandatory office return reduces productivity"
  B2 SOCAVA A1

Rules:
- New arguments use IDs B1, B2, ... continuing from existing B-IDs in the graph
- If you cannot find any new meaningful attacks, output only: SIN_NUEVOS_ATAQUES
- Wrap output in <ATAQUES>...</ATAQUES> tags"""


DEVIL_TEMPLATE = """Problem: {problem}

Current argument graph:
{current_graph}

Generate the strongest possible attacks against the pro-conclusion arguments (those with A-IDs).
For EACH new argument you create, you MUST declare which existing argument it attacks using REFUTA or SOCAVA."""


NARRATOR_SYSTEM = """Translate a formal argument evaluation into a clear, traceable explanation.

Rules:
1. Do NOT add your own reasoning -- only translate what the formal evaluation shows
2. Cite every argument by its ID in brackets, e.g. [A3]
3. Do NOT use hedging language ("seems", "might", "could be")
4. If the conclusion is not justified, say so clearly and explain which attack succeeded

Output structure (use exactly these headers):
CONCLUSION: [text]
ESTADO: Justificada | No justificada | Ambigua

POR QUE SE SOSTIENE:
  [explanation of winning arguments]

POR QUE LOS CONTRA-ARGUMENTOS NO PROSPERARON:
  [explanation of each defeated argument]

TENSION NO RESUELTA: [list or "ninguna"]"""


NARRATOR_TEMPLATE = """Argument graph:
{graph_cnl}

Evaluation result:
- Conclusion: {conclusion}
- Justified: {justified}
- Supporting argument: {supporting_id}
- Grounded extension (accepted): {accepted}
- Defeated: {defeated}
- Undecidable: {undecidable}

Translate this formal evaluation into a traceable explanation."""


print("Modulo 4 OK: todos los prompts definidos")


Modulo 4 OK: todos los prompts definidos


## Modulo 5 -- Clientes LLM

Todos los clientes implementan la misma interfaz: `call(system, user) -> str`

| Cliente | Proveedor | Requiere |
|---------|-----------|----------|
| `AnthropicClient` | Anthropic -- Claude | `ANTHROPIC_API_KEY` |
| `OpenAIClient` | OpenAI -- GPT | `OPENAI_API_KEY` |
| `GeminiClient` | Google -- Gemini | `GEMINI_API_KEY` |
| `OllamaClient` | Ollama -- modelos locales | Ollama server en localhost |
| `HuggingFaceClient` | HuggingFace -- Inference API | `HF_TOKEN` |
| `HuggingFaceLocalClient` | HuggingFace -- GPU local | GPU T4/A100 en Colab |


In [ ]:
from __future__ import annotations
import os
from abc import ABC, abstractmethod


class LLMClient(ABC):
    @abstractmethod
    def call(self, system: str, user: str) -> str: ...


class AnthropicClient(LLMClient):
    # Modelos: "claude-sonnet-4-6" (default), "claude-haiku-4-5-20251001" (rapido)
    def __init__(self, api_key: str | None = None, model: str = "claude-sonnet-4-6"):
        import anthropic
        key = api_key or os.environ.get("ANTHROPIC_API_KEY")
        if not key:
            raise ValueError("Falta ANTHROPIC_API_KEY.")
        self._client = anthropic.Anthropic(api_key=key)
        self._model = model

    def call(self, system: str, user: str) -> str:
        msg = self._client.messages.create(
            model=self._model, max_tokens=2048, system=system,
            messages=[{"role": "user", "content": user}],
        )
        return msg.content[0].text


class OpenAIClient(LLMClient):
    # Modelos: "gpt-4o" (default), "gpt-4o-mini" (economico)
    def __init__(self, api_key: str | None = None, model: str = "gpt-4o"):
        from openai import OpenAI
        key = api_key or os.environ.get("OPENAI_API_KEY")
        if not key:
            raise ValueError("Falta OPENAI_API_KEY.")
        self._client = OpenAI(api_key=key)
        self._model = model

    def call(self, system: str, user: str) -> str:
        response = self._client.chat.completions.create(
            model=self._model,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            max_tokens=2048,
        )
        return response.choices[0].message.content


class GeminiClient(LLMClient):
    # Modelos: "gemini-2.5-flash" (default), "gemini-2.5-pro" (mas potente)
    def __init__(self, api_key: str | None = None, model: str = "gemini-2.5-flash"):
        from google import genai
        key = api_key or os.environ.get("GEMINI_API_KEY")
        if not key:
            raise ValueError("Falta GEMINI_API_KEY. Gratis en https://aistudio.google.com")
        self._client = genai.Client(api_key=key)
        self._model = model

    def call(self, system: str, user: str) -> str:
        from google.genai import types
        response = self._client.models.generate_content(
            model=self._model,
            config=types.GenerateContentConfig(system_instruction=system),
            contents=user,
        )
        return response.text


class OllamaClient(LLMClient):
    # Instalacion en Colab:
    #   !curl -fsSL https://ollama.com/install.sh | sh
    #   import subprocess; subprocess.Popen(["ollama", "serve"])
    #   !ollama pull llama3.2
    def __init__(self, model: str = "llama3.2", base_url: str = "http://localhost:11434/v1"):
        from openai import OpenAI
        self._client = OpenAI(api_key="ollama", base_url=base_url)
        self._model = model

    def call(self, system: str, user: str) -> str:
        response = self._client.chat.completions.create(
            model=self._model,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            max_tokens=2048,
        )
        return response.choices[0].message.content


class HuggingFaceClient(LLMClient):
    # Inference API en la nube. Token gratis en https://huggingface.co/settings/tokens
    #
    # Modelos compatibles con el router de HF (usar exactamente estos nombres):
    #   "Qwen/Qwen2.5-7B-Instruct"             7B, buena calidad (default)
    #   "Qwen/Qwen2.5-72B-Instruct"            72B, muy potente
    #   "meta-llama/Llama-3.2-3B-Instruct"     3B, rapido y liviano
    #   "mistralai/Mistral-Nemo-Instruct-2407"  12B, buena calidad
    #   "microsoft/Phi-3.5-mini-instruct"        3.8B, rapido
    def __init__(self, api_key: str | None = None,
                 model: str = "Qwen/Qwen2.5-7B-Instruct"):
        from huggingface_hub import InferenceClient
        key = api_key or os.environ.get("HF_TOKEN")
        if not key:
            raise ValueError("Falta HF_TOKEN. Gratis en https://huggingface.co/settings/tokens")
        self._client = InferenceClient(api_key=key)
        self._model = model

    def call(self, system: str, user: str) -> str:
        response = self._client.chat_completion(
            model=self._model,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            max_tokens=2048,
        )
        return response.choices[0].message.content


class HuggingFaceLocalClient(LLMClient):
    # Corre un modelo HuggingFace directamente en la GPU del entorno.
    #
    # REQUISITOS:
    #   - GPU habilitada en Colab: Entorno de ejecucion -> Cambiar tipo -> GPU T4
    #   - Las dependencias ya se instalaron en la celda de instalacion (arriba)
    #
    # Primera ejecucion: descarga el modelo (varios GB, 5-15 minutos).
    # Ejecuciones siguientes: carga desde cache en ~30 segundos.
    #
    # Modelos que caben en T4 (15GB VRAM) con cuantizacion 4-bit:
    #   "Qwen/Qwen2.5-7B-Instruct"                  excelente calidad (default)
    #   "meta-llama/Meta-Llama-3.1-8B-Instruct"     muy bueno (requiere aceptar licencia en HF)
    #   "mistralai/Mistral-7B-Instruct-v0.3"         bueno para instrucciones
    #   "microsoft/Phi-3.5-mini-instruct"            3.8B, muy rapido
    #
    # Para Colab Pro con A100 (40GB VRAM):
    #   "Qwen/Qwen2.5-14B-Instruct"
    #   "meta-llama/Llama-3.1-70B-Instruct"         (70B en 4-bit)
    def __init__(
        self,
        model_id: str = "Qwen/Qwen2.5-7B-Instruct",
        load_in_4bit: bool = True,
        max_new_tokens: int = 2048,
    ):
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        import torch

        self._max_new_tokens = max_new_tokens
        self._model_id = model_id

        print(f"Cargando {model_id}...")
        print("(La primera vez descarga el modelo; puede tardar varios minutos)")

        self._tokenizer = AutoTokenizer.from_pretrained(model_id)

        quantization_config = None
        if load_in_4bit:
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )

        self._model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=quantization_config,
            device_map="auto",
            torch_dtype=torch.float16 if not load_in_4bit else None,
        )
        self._model.eval()
        print(f"Modelo listo en: {next(self._model.parameters()).device}")

    def call(self, system: str, user: str) -> str:
        import torch
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        device = next(self._model.parameters()).device

        # apply_chat_template con tokenize=False devuelve el string formateado.
        # Luego tokenizamos por separado para obtener siempre un tensor (no BatchEncoding),
        # lo que evita el AttributeError en transformers >= 4.40.
        chat_text = self._tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False,
        )
        # add_special_tokens=False porque apply_chat_template ya los agrego
        input_ids = self._tokenizer.encode(
            chat_text, return_tensors="pt", add_special_tokens=False,
        ).to(device)
        prompt_len = input_ids.shape[-1]

        with torch.no_grad():
            outputs = self._model.generate(
                input_ids,
                max_new_tokens=self._max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=self._tokenizer.eos_token_id,
            )
        new_tokens = outputs[0][prompt_len:]
        return self._tokenizer.decode(new_tokens, skip_special_tokens=True)


def call_with_retry(
    client: LLMClient,
    system: str,
    user: str,
    tag: str,
    known_ids: set,
    max_retries: int = 2,
) -> str:
    # Reintentos automaticos si el CNL generado tiene errores de formato.
    current_user = user
    last_block = ""
    for attempt in range(max_retries + 1):
        raw = client.call(system, current_user)
        block = extract_tagged_block(raw, tag)
        errors = validate_block(block, known_ids)
        last_block = block
        if not errors:
            return block
        if attempt < max_retries:
            current_user = (
                user + f"\n\nPrevious response had formatting errors: {errors}. "
                "Please fix them and respond again with the corrected CNL."
            )
    return last_block


print("Modulo 5 OK: LLMClient, AnthropicClient, OpenAIClient, GeminiClient, OllamaClient,")
print("             HuggingFaceClient, HuggingFaceLocalClient, call_with_retry")


Modulo 5 OK: LLMClient, AnthropicClient, OpenAIClient, GeminiClient, OllamaClient,
             HuggingFaceClient, HuggingFaceLocalClient, call_with_retry


## Modulo 6 -- Generador de Argumentos (C1)

- Primera llamada (`graph=None`): genera argumentos iniciales y define la conclusion target.
- Llamadas siguientes: genera argumentos para rebatir ataques B, continuando la numeracion A-ID.


In [ ]:
from __future__ import annotations


def generate_arguments(
    problem: str,
    client: LLMClient,
    graph: ArgumentGraph | None = None,
) -> tuple[list, list, str | None]:
    is_initial = graph is None
    known_ids  = set() if is_initial else set(graph.arguments.keys())

    if is_initial:
        user = GENERATOR_INITIAL_TEMPLATE.format(problem=problem)
    else:
        user = GENERATOR_RESPONSE_TEMPLATE.format(
            problem=problem, current_graph=graph.to_cnl()
        )

    block = call_with_retry(client, GENERATOR_SYSTEM, user, "ARGUMENTOS", known_ids)
    args, attacks, conclusion = parse_cnl_block(block)
    return args, attacks, (conclusion if is_initial else None)


print("Modulo 6 OK: generate_arguments")


Modulo 6 OK: generate_arguments


## Modulo 7 -- Abogado del Diablo (C2)

Genera los mejores contra-argumentos posibles (steel-manning).
Si no puede encontrar ataques significativos, responde `SIN_NUEVOS_ATAQUES`.


In [ ]:
from __future__ import annotations


def generate_attacks(
    problem: str,
    graph: ArgumentGraph,
    client: LLMClient,
) -> tuple[list, list, bool]:
    # Devuelve (nuevos_argumentos_B, nuevos_ataques, exhausted).
    # exhausted=True significa que el loop debe terminar.
    user = DEVIL_TEMPLATE.format(problem=problem, current_graph=graph.to_cnl())
    block = call_with_retry(client, DEVIL_SYSTEM, user, "ATAQUES", set())

    if not block.strip() or "SIN_NUEVOS_ATAQUES" in block:
        return [], [], True

    args, attacks, _ = parse_cnl_block(block)

    # Descartar ataques cuyo atacante no existe (ni en el grafo ni entre los nuevos B-IDs)
    valid_ids = set(graph.arguments.keys()) | {a.id for a in args}
    attacks   = [atk for atk in attacks if atk.attacker_id in valid_ids]

    return args, attacks, False


print("Modulo 7 OK: generate_attacks")


Modulo 7 OK: generate_attacks


## Modulo 8 -- Loop Adversarial

Orquesta el dialogo C1 <-> C2 para construir el grafo.

Termina cuando:
1. C2 emite `SIN_NUEVOS_ATAQUES`
2. Se alcanza `max_rounds`
3. El grafo no crece en la ultima ronda (sin progreso)


In [ ]:
from __future__ import annotations


def _graph_size(graph: ArgumentGraph) -> int:
    return len(graph.arguments) + len(graph.attacks)


def _merge_into(graph: ArgumentGraph, args: list, attacks: list) -> None:
    for a in args:
        graph.arguments[a.id] = a
    graph.attacks.extend(attacks)


def run_adversarial_loop(
    problem: str,
    client: LLMClient,
    max_rounds: int = 3,
    verbose: bool = False,
) -> ArgumentGraph:
    def _log(msg: str) -> None:
        if verbose:
            print(f"  [loop] {msg}")

    args, attacks, target_conclusion = generate_arguments(problem, client, graph=None)
    graph = ArgumentGraph(
        arguments={a.id: a for a in args},
        attacks=list(attacks),
        target_conclusion=target_conclusion or "",
    )
    _log(f"ronda 0 -- generador: {len(args)} args, {len(attacks)} ataques")

    prev_size = _graph_size(graph)
    for round_num in range(max_rounds):
        new_args, new_attacks, exhausted = generate_attacks(problem, graph, client)
        _log(f"ronda {round_num+1} -- diablo: {len(new_args)} args, {len(new_attacks)} ataques, agotado={exhausted}")
        _merge_into(graph, new_args, new_attacks)

        if exhausted or _graph_size(graph) == prev_size:
            break
        prev_size = _graph_size(graph)

        new_args, new_attacks, _ = generate_arguments(problem, client, graph=graph)
        _log(f"ronda {round_num+1} -- generador (respuesta): {len(new_args)} args, {len(new_attacks)} ataques")
        _merge_into(graph, new_args, new_attacks)

        if _graph_size(graph) == prev_size:
            break
        prev_size = _graph_size(graph)

    return graph


print("Modulo 8 OK: run_adversarial_loop")


Modulo 8 OK: run_adversarial_loop


## Modulo 9 -- Narrador (C5)

Traduce el resultado formal a lenguaje natural trazable.
No agrega razonamiento propio: solo traduce lo que el grafo y la evaluacion muestran.


In [ ]:
from __future__ import annotations


def narrate(graph: ArgumentGraph, result: EvaluationResult, client: LLMClient) -> str:
    user = NARRATOR_TEMPLATE.format(
        graph_cnl=graph.to_cnl(),
        conclusion=result.conclusion,
        justified=result.justified,
        supporting_id=result.supporting_argument_id or "ninguno",
        accepted=result.grounded_extension,
        defeated=result.defeated,
        undecidable=result.undecidable,
    )
    return client.call(NARRATOR_SYSTEM, user)


print("Modulo 9 OK: narrate")


Modulo 9 OK: narrate


## Modulo 10 -- Pipeline

Orquesta todos los componentes en 3 pasos:

1. `run_adversarial_loop` -> construye el grafo argumentativo (C1 <-> C2)
2. `grounded_extension` -> evalua que argumentos sobreviven (motor Dung)
3. `narrate` -> genera la explicacion en lenguaje natural (C5)


In [ ]:
from __future__ import annotations
import json as _json


class Pipeline:
    # Orquesta el flujo completo de argumentacion formal.
    #
    # Uso basico:
    #   client   = GeminiClient(api_key="tu-key")
    #   pipeline = Pipeline(client=client, max_rounds=2)
    #   result   = pipeline.run("Es etico el trabajo remoto obligatorio?")
    #   print(result.explanation)

    def __init__(self, client: LLMClient, max_rounds: int = 3, verbose: bool = False):
        self._client     = client
        self._max_rounds = max_rounds
        self._verbose    = verbose

    def run(self, problem: str) -> PipelineResult:
        if self._verbose:
            print(f"\nAnalizando: {problem[:80]}...")
            print()

        graph = run_adversarial_loop(
            problem, self._client, self._max_rounds, verbose=self._verbose
        )

        if self._verbose:
            print(f"\nGrafo final: {len(graph.arguments)} argumentos, {len(graph.attacks)} ataques")
            print("Evaluando extension fundamentada (Dung)...")

        evaluation = grounded_extension(graph)

        if self._verbose:
            estado = "JUSTIFICADA" if evaluation.justified else "NO JUSTIFICADA"
            print(f"{estado}  |  Soporte: {evaluation.supporting_argument_id}  |  Derrotados: {evaluation.defeated}")
            print("Generando explicacion...")

        explanation = narrate(graph, evaluation, self._client)
        return PipelineResult(explanation=explanation, graph=graph, evaluation=evaluation)

    def save(self, result: PipelineResult, path: str) -> None:
        data = {
            "conclusion":             result.evaluation.conclusion,
            "justified":              result.evaluation.justified,
            "supporting_argument_id": result.evaluation.supporting_argument_id,
            "grounded_extension":     result.evaluation.grounded_extension,
            "defeated":               result.evaluation.defeated,
            "undecidable":            result.evaluation.undecidable,
            "explanation":            result.explanation,
            "graph_cnl":              result.graph.to_cnl(),
        }
        with open(path, "w", encoding="utf-8") as f:
            _json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Resultado guardado en: {path}")


print("Modulo 10 OK: Pipeline")
print()
print("Sistema completo. Todos los modulos cargados:")
for mod in ["Argument/Attack/ArgumentGraph/EvaluationResult/PipelineResult",
            "parse_cnl_block / validate_block / extract_tagged_block",
            "grounded_extension",
            "Prompts (GENERATOR, DEVIL, NARRATOR)",
            "LLMClient / AnthropicClient / OpenAIClient / GeminiClient",
            "OllamaClient / HuggingFaceClient / HuggingFaceLocalClient",
            "generate_arguments", "generate_attacks",
            "run_adversarial_loop", "narrate", "Pipeline"]:
    print(f"  {mod}")


Modulo 10 OK: Pipeline

Sistema completo. Todos los modulos cargados:
  Argument/Attack/ArgumentGraph/EvaluationResult/PipelineResult
  parse_cnl_block / validate_block / extract_tagged_block
  grounded_extension
  Prompts (GENERATOR, DEVIL, NARRATOR)
  LLMClient / AnthropicClient / OpenAIClient / GeminiClient
  OllamaClient / HuggingFaceClient / HuggingFaceLocalClient
  generate_arguments
  generate_attacks
  run_adversarial_loop
  narrate
  Pipeline


## Configuracion -- Elige tu proveedor LLM

Ejecuta **solo una** de las siguientes celdas.

> Si algun nombre da `NameError`, ejecuta primero **Runtime > Ejecutar todo** (Ctrl+F9).


### Opcion A -- Anthropic Claude

Key en https://console.anthropic.com

In [ ]:
ANTHROPIC_API_KEY = "sk-ant-..."   # reemplaza con tu key

client = AnthropicClient(
    api_key=ANTHROPIC_API_KEY,
    model="claude-sonnet-4-6",
)
print("Usando Anthropic Claude:", client._model)


Usando Anthropic Claude: claude-sonnet-4-6


### Opcion B -- OpenAI GPT

Key en https://platform.openai.com/api-keys

In [ ]:
OPENAI_API_KEY = "sk-proj-..."   # reemplaza con tu key

client = OpenAIClient(
    api_key=OPENAI_API_KEY,
    model="gpt-4o",
)
print("Usando OpenAI:", client._model)


Usando OpenAI: gpt-4o


### Opcion C -- Google Gemini

Key gratuita en https://aistudio.google.com (~20 req/dia)

In [ ]:
GEMINI_API_KEY = "AIza..."   # reemplaza con tu key

client = GeminiClient(
    api_key=GEMINI_API_KEY,
    model="gemini-2.5-flash",
)
print("Usando Google Gemini:", client._model)


Usando Google Gemini: gemini-2.5-flash


### Opcion D -- Ollama (local, 100% gratis)

```bash
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess; subprocess.Popen(["ollama", "serve"])
!ollama pull llama3.2
```


In [ ]:
client = OllamaClient(
    model="llama3.2",
    base_url="http://localhost:11434/v1",
)
print("Usando Ollama:", client._model)


Usando Ollama: llama3.2


### Opcion E -- HuggingFace Inference API (nube, tier gratuito)

Token gratuito en https://huggingface.co/settings/tokens

Modelos compatibles con el router de HF:
- `Qwen/Qwen2.5-7B-Instruct` (default, buena calidad)
- `Qwen/Qwen2.5-72B-Instruct` (muy potente)
- `meta-llama/Llama-3.2-3B-Instruct` (rapido)
- `mistralai/Mistral-Nemo-Instruct-2407` (12B)


In [ ]:
HF_TOKEN = "hf_..."   # reemplaza con tu token

client = HuggingFaceClient(
    api_key=HF_TOKEN,
    model="Qwen/Qwen2.5-7B-Instruct",
)
print("Usando HuggingFace API:", client._model)


Usando HuggingFace API: Qwen/Qwen2.5-7B-Instruct


### Opcion F -- HuggingFace Local (GPU de Colab, sin rate limits)

**Requisito:** activar GPU antes de ejecutar.
Colab: Entorno de ejecucion -> Cambiar tipo de entorno -> GPU T4 (gratis)

Primera ejecucion: descarga el modelo (~14GB para 7B en 4-bit, tarda 5-15 min).
Ejecuciones siguientes: carga desde cache en ~30 segundos.


In [ ]:
client = HuggingFaceLocalClient(
    model_id="Qwen/Qwen2.5-7B-Instruct",   # 7B, excelente calidad, cabe en T4
    # model_id="microsoft/Phi-3.5-mini-instruct",  # 3.8B, mas rapido
    # model_id="meta-llama/Meta-Llama-3.1-8B-Instruct",  # 8B (requiere licencia HF)
    load_in_4bit=True,
)
print("Usando HuggingFace Local:", client._model_id)


Cargando Qwen/Qwen2.5-7B-Instruct...
(La primera vez descarga el modelo; puede tardar varios minutos)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modelo listo en: cuda:0
Usando HuggingFace Local: Qwen/Qwen2.5-7B-Instruct


## A opinar que es gratis!

Modifica `PREGUNTA` con el tema que quieras analizar.
- `max_rounds=2` -> aprox 7 llamadas LLM (recomendado para empezar)
- `verbose=True` -> muestra el progreso ronda a ronda


In [ ]:
PREGUNTA = "The United States should not forcibly capture Nicolás Maduro, even if it accuses him of drug trafficking and terrorism, because doing so against a foreign leader may violate Venezuela’s sovereignty, degrade international law, and set a dangerous precedent."

# Otras preguntas de ejemplo:
# PREGUNTA = "Es etico usar inteligencia artificial para tomar decisiones judiciales?"
# PREGUNTA = "Deberia el voto ser obligatorio en una democracia?"
# PREGUNTA = "Es el capitalismo el mejor sistema economico posible?"
# PREGUNTA = "Should social media platforms be regulated by governments?"

pipeline = Pipeline(
    client=client,
    max_rounds=2,
    verbose=True,
)

print("=" * 65)
print("PREGUNTA:", PREGUNTA)
print("=" * 65)

result = pipeline.run(PREGUNTA)

print()
print("=" * 65)
print("ANALISIS FORMAL")
print("=" * 65)
print()
print(result.explanation)
print()
print("-" * 65)
print("Argumentos aceptados :", result.evaluation.grounded_extension)
print("Argumentos derrotados:", result.evaluation.defeated)
print("Indecidibles         :", result.evaluation.undecidable or "ninguno")
print("Justificada          :", result.evaluation.justified)


PREGUNTA: The United States should not forcibly capture Nicolás Maduro, even if it accuses him of drug trafficking and terrorism, because doing so against a foreign leader may violate Venezuela’s sovereignty, degrade international law, and set a dangerous precedent.

Analizando: The United States should not forcibly capture Nicolás Maduro, even if it accuses...

  [loop] ronda 0 -- generador: 3 args, 0 ataques
  [loop] ronda 1 -- diablo: 3 args, 3 ataques, agotado=False
  [loop] ronda 1 -- generador (respuesta): 3 args, 3 ataques
  [loop] ronda 2 -- diablo: 7 args, 7 ataques, agotado=False
  [loop] ronda 2 -- generador (respuesta): 10 args, 10 ataques

Grafo final: 26 argumentos, 23 ataques
Evaluando extension fundamentada (Dung)...
JUSTIFICADA  |  Soporte: A1  |  Derrotados: ['B1', 'B10', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9']
Generando explicacion...

ANALISIS FORMAL

CONCLUSIÓN: "The United States should not forcibly capture Nicolás Maduro."
ESTADO: Justificada

POR QUE SE 

## (Opcional) Guardar resultado

In [ ]:
# Guarda el resultado completo en JSON para auditoria o analisis posterior.
pipeline.save(result, "hablemos_sin_saber.json")


Resultado guardado en: venezuela.json
